# Single-Stock Return Analysis - Meta Plafroms (2012-2026)

**Question:** What do the return, volatility, and drawdown characteristics of a single large-cap equity look like - and how far do they depart from the normal distribution assumed by introductory models? 

**Data:** Yahoo Finance daily prices via 'yfinance' ('auto_adjust=True', so splits and dividends are folded in - daily percent change approximates a total return).
Downloaded once and cahed to 'data/'; the analysis reads from disk.

**Author:** Alexandre • [Github](https://github.com/alexandret-11) • [LinkedIn](https://linkedin.com/in/alexandrethompson)

---
*Project 1 of a 7-projext quantitative research portfolio. Next: multi-asset comparison and correlation structure*

## Setup

Interim data source qhile WRDS/CRSP access is pending. The pull is isolated in a singkle cell so that the data layer can be swapped for CRSP later without touching any analysis code.

## Data

One network call, then cached. Two data-handling detials worth flagging: newer 'yfinance' versions return two-level columns (flattened at the boundary below), and the first row's return is 'NaN by construction - a series of *n* prices yields *n-1* returns.

In [10]:
# Block 0 - Data Pull(Yahoo Finance)
# Interim source while WRDS summer access is pending

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# auto_adjust=True -> closes adjusted for splits AND dividends, so pct_change
# approximates a total return
raw = yf.download(
    "META",
    start="2012-05-18",  # IPO date
    auto_adjust=True,
    progress=False,
)

# Newer yfinance returns MultiIndex columns (built for multi-ticker calls);
# flatten so raw["Close"] works across versions
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

# .copy() serves df from raw -> avoids SettingWithCopyWarning on assignment
df = raw[["Close"]].rename(columns={"Close": "px_adj"}).copy()
df["ret"] = df["px_adj"].pct_change()
df.to_csv("../data/meta_prices_yf.csv")

# Four Diagnostic Tests
print(df.shape)
print(df.head(3))
print(df.tail(3))
print(df["ret"].isna().sum())

(3585, 2)
Price          px_adj       ret
Date                           
2012-05-18  37.897202       NaN
2012-05-21  33.733765 -0.109861
2012-05-22  30.730141 -0.089039
Price           px_adj       ret
Date                            
2026-08-19  546.030029  0.004341
2026-08-20  545.830017 -0.000366
2026-08-21  549.900024  0.007457
1


In [ ]:
df["log"] = np.log(df["px_adj"]).diff()

print(df[["ret", "log"]].describe())

gap = (df["ret"] - df["log"]).abs()
print(df.loc[gap.nlargest(5).index, ["ret", "log"]])

print(
    f"Worst day: {df['ret'].idxmin().date()} "
    f"simple {df['ret'].min():.2%}, "
    f"log {df['log'].min():.2%}"
)

SyntaxError: closing parenthesis ')' does not match opening parenthesis '{' on line 11 (2628637742.py, line 12)